[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C37_MLOps_Course/05_feedback_retraining/05_feedback_retraining.ipynb)

# 05 · 从零模拟反馈闭环与重训练

本 notebook 把整条闭环跑起来：**反馈聚合（区分置信度）+ 主动学习选样 + 重训触发器（持续性+冷却）+ 灾难性遗忘度量 + 回放修复 + 完整闭环模拟**，用 **numpy + 标准库** 实现，过 `assert`。这是全课的收官——把五个模块拼成一个转动的环。

**路线**：① 聚合带置信度的反馈 → ② 主动学习挑样本 → ③ 重训触发器 → ④ 灾难性遗忘 + 回放修复 → ⑤ 端到端闭环 → ✏️ 4 道练习 → 📖 答案 → 🧪 完整生命周期闭环模拟。

In [ ]:
import numpy as np
from collections import deque
rng = np.random.default_rng(0)
print('环境就绪 ✅ | numpy', np.__version__)

## 1 · 聚合反馈：直接信号压倒间接信号

同一个样本可能收到多条反馈（多个用户、多种信号）。聚合时**直接反馈（高置信）应压倒间接反馈（低置信）**。

实现一个加权投票：每条反馈带 (标签, 置信度权重)，按权重投票得聚合标签。

In [ ]:
def aggregate_feedback(feedbacks):
    '''feedbacks: [(label, weight), ...]；返回 (聚合标签, 总权重, 是否高置信)。'''
    if not feedbacks:
        return None, 0.0, False
    score = {}
    for label, w in feedbacks:
        score[label] = score.get(label, 0.0) + w
    total = sum(w for _, w in feedbacks)
    best_label = max(score, key=score.get)
    confidence = score[best_label] / total           # 胜出标签占的权重比例
    return best_label, total, confidence >= 0.7

# 权重约定：人工标注=1.0(直接), 点击=0.2(间接弱信号)
# 3 个间接「正」(共0.6) vs 1 个直接「负」(1.0) -> 直接信号应压倒
fb = [(1, 0.2), (1, 0.2), (1, 0.2), (0, 1.0)]
label, total, high_conf = aggregate_feedback(fb)
print(f'聚合标签={label}, 总权重={total:.1f}, 高置信={high_conf}')
assert label == 0, '一条直接反馈(1.0)应压倒三条间接(共0.6)'
# 全是一致的直接反馈 -> 高置信
label2, _, hc2 = aggregate_feedback([(1, 1.0), (1, 1.0)])
assert label2 == 1 and hc2 == True, '一致的直接反馈应高置信'
# 五五开的弱信号 -> 低置信
_, _, hc3 = aggregate_feedback([(1, 0.2), (0, 0.2)])
assert hc3 == False, '势均力敌应低置信'
print('✅ 反馈聚合正确：直接信号压倒间接，置信度反映共识强度')

## 2 · 主动学习：让模型挑「最该问人」的样本

标注最贵。主动学习：优先标注**模型最不确定**的样本（信息量最大）。

对二分类，预测概率最接近 0.5 的样本最不确定。实现 `select_for_labeling(probs, budget)`：选最不确定的 `budget` 个样本下标。

In [ ]:
def uncertainty(prob):
    # 二分类不确定性：越接近 0.5 越不确定。用 -|p-0.5| 排序即可
    return -np.abs(prob - 0.5)

def select_for_labeling(probs, budget):
    '''返回最不确定的 budget 个样本下标（按不确定性降序）。'''
    probs = np.asarray(probs)
    scores = uncertainty(probs)
    return np.argsort(scores)[::-1][:budget]

probs = np.array([0.99, 0.51, 0.50, 0.02, 0.48, 0.95])
idx = select_for_labeling(probs, budget=3)
print('预测概率:', probs)
print('选去标注的下标:', idx, '-> 概率', probs[idx])
# 最该标的是 0.50/0.51/0.48（最接近 0.5），而非 0.99/0.02（模型很确定）
assert set(idx.tolist()) == {1, 2, 4}, '应选最接近 0.5 的三个'
assert 0 not in idx and 3 not in idx, '高置信样本(0.99,0.02)不该浪费标注预算'
# 对比：主动学习选的样本平均不确定性 > 随机选
active_unc = uncertainty(probs[idx]).mean()
random_unc = uncertainty(probs).mean()
assert active_unc > random_unc, '主动学习选的样本应比随机更不确定（信息量更大）'
print('✅ 主动学习：把宝贵的标注预算花在模型最拿不准的样本上')

## 3 · 重训触发器：持续性 + 冷却期

触发条件组合：漂移超阈 **或** 性能跌破 SLO **或** 攒够 N 条新数据 **或** 定时。

防抖动：要求**连续 k 步**满足才触发（持续性），且刚重训完进入**冷却期**不再触发。

In [ ]:
class RetrainTrigger:
    def __init__(self, psi_thresh=0.25, perf_slo=0.85, data_thresh=1000,
                 time_thresh=30, persistence=2, cooldown=3):
        self.psi_thresh=psi_thresh; self.perf_slo=perf_slo
        self.data_thresh=data_thresh; self.time_thresh=time_thresh
        self.persistence=persistence; self.cooldown=cooldown
        self.consec=0; self.cooldown_left=0; self.steps_since_retrain=0
    def step(self, psi, perf, new_data_count):
        '''每个监控周期调一次；返回 (是否触发重训, 原因)。'''
        self.steps_since_retrain += 1
        if self.cooldown_left > 0:                 # 冷却中，不触发
            self.cooldown_left -= 1
            self.consec = 0
            return False, 'cooldown'
        reasons = []
        if psi >= self.psi_thresh: reasons.append('drift')
        if perf < self.perf_slo: reasons.append('perf<SLO')
        if new_data_count >= self.data_thresh: reasons.append('enough_data')
        if self.steps_since_retrain >= self.time_thresh: reasons.append('scheduled')
        if reasons:
            self.consec += 1
        else:
            self.consec = 0
        if self.consec >= self.persistence:        # 持续满足才真触发
            self.consec = 0
            self.cooldown_left = self.cooldown
            self.steps_since_retrain = 0
            return True, '+'.join(reasons)
        return False, ('pending:' + '+'.join(reasons) if reasons else 'ok')

trig = RetrainTrigger(persistence=2, cooldown=3)
# 模拟一串监控读数：单次尖峰不触发，持续漂移才触发
stream = [(0.05,0.92,10),(0.30,0.91,20),(0.06,0.92,30),   # 第2步尖峰，但不持续
          (0.28,0.90,40),(0.29,0.89,50)]                   # 第4,5步持续漂移 -> 触发
fired = []
for i,(psi,perf,nd) in enumerate(stream):
    f, why = trig.step(psi, perf, nd)
    print(f'step {i}: psi={psi} perf={perf} -> {"RETRAIN" if f else "hold"} ({why})')
    if f: fired.append(i)
assert 1 not in fired, '单次尖峰(step1)不应触发（持续性过滤）'
assert 4 in fired, '持续漂移(step3,4)应在 step4 触发'
print('✅ 触发器正确：持续性过滤尖峰，触发后进入冷却')

## 4 · 灾难性遗忘 + 回放修复

增量学习的深坑：在新任务上接着训，会**急剧丢失旧任务能力**。我们用一个玩具线性分类器复现它，再用 **replay（混入旧数据）** 修复。

任务 A、任务 B 是两个不同的线性可分问题。先学 A，再只用 B 数据继续训 -> 测 A 的准确率会暴跌。

In [ ]:
def make_task(w_true, n, seed):
    r = np.random.default_rng(seed)
    X = r.normal(0, 1, (n, 3))
    y = (X @ w_true > 0).astype(float)
    return X, y

def train_logistic(X, y, w_init=None, lr=0.5, epochs=300):
    w = np.zeros(X.shape[1]) if w_init is None else w_init.copy()
    for _ in range(epochs):
        p = 1/(1+np.exp(-(X @ w)))
        grad = X.T @ (p - y) / len(y)
        w -= lr * grad
    return w

def acc(w, X, y):
    return float(((X @ w > 0).astype(float) == y).mean())

# 两个差异很大的任务（权重方向几乎相反）
wA = np.array([3.0, 1.0, -1.0]); wB = np.array([-1.0, 3.0, 2.0])
XA, yA = make_task(wA, 800, 1); XA_te, yA_te = make_task(wA, 400, 2)
XB, yB = make_task(wB, 800, 3)

# 阶段1：学任务 A
w = train_logistic(XA, yA)
acc_A_before = acc(w, XA_te, yA_te)
print(f'学完 A 后，A 测试集准确率 = {acc_A_before:.3f}')
assert acc_A_before > 0.9, 'A 应学得好'

In [ ]:
# 阶段2a：只用 B 数据接着训（朴素增量）-> 灾难性遗忘
w_naive = train_logistic(XB, yB, w_init=w)
acc_A_after_naive = acc(w_naive, XA_te, yA_te)
forgetting_naive = acc_A_before - acc_A_after_naive
print(f'只学 B 后，A 准确率 = {acc_A_after_naive:.3f}  (遗忘 = {forgetting_naive:.3f})')

# 阶段2b：B 数据 + 回放一批 A 数据（replay）-> 抑制遗忘
replay_idx = rng.integers(0, len(XA), 400)              # 抽 400 条旧数据回放
X_mix = np.vstack([XB, XA[replay_idx]])
y_mix = np.concatenate([yB, yA[replay_idx]])
w_replay = train_logistic(X_mix, y_mix, w_init=w)
acc_A_after_replay = acc(w_replay, XA_te, yA_te)
forgetting_replay = acc_A_before - acc_A_after_replay
print(f'B+回放后，A 准确率 = {acc_A_after_replay:.3f}  (遗忘 = {forgetting_replay:.3f})')

assert forgetting_naive > 0.15, '朴素增量应造成明显的灾难性遗忘'
assert forgetting_replay < forgetting_naive, '回放应显著抑制遗忘'
print(f'\n✅ 复现灾难性遗忘(掉{forgetting_naive:.0%})，并用 replay 抑制(掉{forgetting_replay:.0%})')

## 5 · 端到端闭环：模型一代代进化

把前面的零件拼成一个**转动的闭环状态机**：环境漂移 → 监控测漂移 → 触发器决定是否重训 → 重训(含回放) → 评估 → 更新。

跑几轮，看模型在漂移的世界里靠重训**维持**性能（vs 不重训则持续劣化）。

In [ ]:
def env_data(t, n, seed):
    '''随时间 t 漂移的环境：决策边界随 t 缓慢旋转。'''
    r = np.random.default_rng(seed)
    angle = t * 0.25                                # 每步旋转一点（概念漂移）
    w_t = np.array([np.cos(angle), np.sin(angle), 0.5])
    X = r.normal(0, 1, (n, 3))
    y = (X @ w_t > 0).astype(float)
    return X, y

# 初始模型在 t=0 训练
X0, y0 = env_data(0, 800, 100)
w_model = train_logistic(X0, y0)
ref0, _ = env_data(0, 1000, 101)                    # 漂移检测参考

def psi_1d(ref, cur, bins=10, eps=1e-6):
    edges = np.quantile(ref, np.linspace(0,1,bins+1)); edges[0],edges[-1]=-np.inf,np.inf
    q=np.clip(np.histogram(ref,edges)[0]/len(ref),eps,None)
    p=np.clip(np.histogram(cur,edges)[0]/len(cur),eps,None)
    return float(np.sum((p-q)*np.log(p/q)))

trigger = RetrainTrigger(perf_slo=0.85, persistence=1, cooldown=1, psi_thresh=999, time_thresh=999)
history_w_retrain = []
data_buffer_X, data_buffer_y = [X0], [y0]
print(f"{'步':<4}{'真实性能':>10}{'触发重训':>10}")
for t in range(1, 8):
    Xt, yt = env_data(t, 800, 100+t)               # 当前(漂移后)环境
    perf = acc(w_model, Xt, yt)                     # 线上真实性能(有标签时)
    fire, why = trigger.step(psi=0.0, perf=perf, new_data_count=0)  # 这里用性能驱动
    if fire:
        data_buffer_X.append(Xt); data_buffer_y.append(yt)         # 攒入新数据
        Xall = np.vstack(data_buffer_X[-3:]); yall = np.concatenate(data_buffer_y[-3:])  # 含回放
        w_model = train_logistic(Xall, yall)        # 重训(混合近几轮数据=自带回放)
    history_w_retrain.append(acc(w_model, Xt, yt))
    print(f'{t:<4}{perf:>10.3f}{("RETRAIN "+why) if fire else "hold":>10}')

# 对照：完全不重训，模型在漂移世界里持续劣化
history_no_retrain = []
for t in range(1, 8):
    Xt, yt = env_data(t, 800, 100+t)
    history_no_retrain.append(acc(w_model if False else train_logistic(X0,y0), Xt, yt))

print(f'\n带重训   末期性能 = {history_w_retrain[-1]:.3f}')
print(f'不重训   末期性能 = {history_no_retrain[-1]:.3f}')
assert history_w_retrain[-1] > history_no_retrain[-1] + 0.05, '重训应在漂移世界里维持更高性能'
assert history_no_retrain[-1] < history_no_retrain[0], '不重训应随漂移持续劣化'
print('✅ 闭环转起来了：重训让模型跟上漂移的世界；不重训则持续劣化')

---
## ✏️ 练习 1：反馈的多数投票 + 平局处理

实现 `majority_vote(labels, tie_break='abstain')`：对一组标签做多数投票。
平局时按 `tie_break`：`'abstain'` 返回 `None`（弃权，交人工），`'positive'` 返回 1。

In [ ]:
def majority_vote(labels, tie_break='abstain'):
    # TODO: 统计各标签票数；取最多者；若最高票有并列(平局)，按 tie_break 处理
    #       'abstain'->None, 'positive'->1
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert majority_vote([1,1,1,0]) == 1, '明显多数'
assert majority_vote([1,0]) is None, '平局默认弃权'
assert majority_vote([1,0,1,0], tie_break='positive') == 1, '平局可配置偏正'
assert majority_vote([0,0,0]) == 0
print('✅ 练习 1 通过：多数投票 + 平局处理')

## ✏️ 练习 2：组合式重训决策

实现 `should_retrain(psi, perf, days_since, n_new)`：满足**任一**条件即返回 `(True, 原因列表)`：
PSI ≥ 0.25 / perf < 0.85 / days_since ≥ 30 / n_new ≥ 1000。都不满足返回 `(False, [])`。

In [ ]:
def should_retrain(psi, perf, days_since, n_new):
    # TODO: 逐条检查四个条件，收集满足的原因字符串；任一满足则 (True, reasons)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
ok, why = should_retrain(0.30, 0.92, 5, 100)
assert ok and 'drift' in str(why).lower().replace('psi','drift') or 'psi' in str(why).lower(), '漂移应触发'
ok2, why2 = should_retrain(0.05, 0.80, 5, 100)
assert ok2, '性能跌破 SLO 应触发'
ok3, why3 = should_retrain(0.05, 0.92, 5, 100)
assert not ok3 and why3 == [], '都不满足则不触发'
ok4, why4 = should_retrain(0.30, 0.80, 40, 2000)
assert ok4 and len(why4) == 4, '多条件同时满足应全部列出'
print('✅ 练习 2 通过：组合式重训决策')

## ✏️ 练习 3：量化灾难性遗忘

实现 `forgetting_score(acc_before, acc_after)`：返回遗忘度量 = 旧任务准确率的**跌幅**（before - after），并 clip 到 ≥ 0（性能不降则遗忘为 0，提升不算「负遗忘」）。

In [ ]:
def forgetting_score(acc_before, acc_after):
    # TODO: 返回 max(0, acc_before - acc_after)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert abs(forgetting_score(0.95, 0.60) - 0.35) < 1e-9, '掉了 0.35'
assert forgetting_score(0.90, 0.93) == 0.0, '没降(反升)则遗忘为 0'
assert forgetting_score(0.88, 0.88) == 0.0, '持平则遗忘为 0'
# 用 worked 里的真实数据验证
f_naive = forgetting_score(acc_A_before, acc_A_after_naive)
f_replay = forgetting_score(acc_A_before, acc_A_after_replay)
assert f_naive > f_replay, '朴素增量遗忘应大于回放'
print(f'朴素增量遗忘={f_naive:.3f}, 回放遗忘={f_replay:.3f}')
print('✅ 练习 3 通过：量化灾难性遗忘')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def majority_vote(labels, tie_break='abstain'):
    from collections import Counter
    cnt = Counter(labels)
    top = max(cnt.values())
    winners = [k for k, v in cnt.items() if v == top]
    if len(winners) == 1:
        return winners[0]
    return 1 if tie_break == 'positive' else None

In [ ]:
# 练习 2 参考答案
def should_retrain(psi, perf, days_since, n_new):
    reasons = []
    if psi >= 0.25: reasons.append('psi_drift')
    if perf < 0.85: reasons.append('perf_below_slo')
    if days_since >= 30: reasons.append('scheduled')
    if n_new >= 1000: reasons.append('enough_data')
    return (len(reasons) > 0, reasons)

In [ ]:
# 练习 3 参考答案
def forgetting_score(acc_before, acc_after):
    return max(0.0, acc_before - acc_after)

---
## 🧪 真实数据胶囊：完整生命周期闭环模拟

把**全部五个模块**拼进一个模拟：一个在持续漂移的世界里运行的分类器，经历**追踪(01) → 版本(02) → 监控漂移(04) → 触发重训(05) → 门禁(03) → 晋升或留守** 的完整循环，跑 12 个周期。

你会看到一条「**监控驱动重训、门禁守住质量**」的生命线：漂移触发重训，但只有过了门禁(显著更好)的新模型才被晋升。

In [ ]:
# === 全课五模块的极简集成 ===
def content_hash(obj):
    import hashlib, json
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode()).hexdigest()[:8]

def bootstrap_better(correct_new, correct_old, B=500, seed=0):
    # 模块03：新模型是否 bootstrap-显著优于旧 (CI 下界>0)
    r = np.random.default_rng(seed); n = len(correct_new)
    deltas = np.array([correct_new[r.integers(0,n,n)].mean()-correct_old[r.integers(0,n,n)].mean() for _ in range(B)])
    return np.percentile(deltas, 2.5) > 0

registry = {}          # 模块02：版本库 hash->model
tracking = []          # 模块01：每轮 run 记录
champion = train_logistic(*env_data(0, 800, 500))
champion_hash = content_hash(np.round(champion,4).tolist())
registry[champion_hash] = champion
ref_feat, _ = env_data(0, 1000, 501)
trig = RetrainTrigger(perf_slo=0.82, persistence=1, cooldown=1, psi_thresh=0.25, time_thresh=999)
buf_X, buf_y = [env_data(0,800,500)[0]], [env_data(0,800,500)[1]]
n_retrains = n_promotions = 0

In [ ]:
print(f"{'周期':<5}{'性能':>8}{'PSI':>8}{'触发':>7}{'门禁':>10}{'champion':>11}")
for t in range(1, 13):
    Xt, yt = env_data(t, 800, 500+t)
    perf = acc(champion, Xt, yt)                              # 真实性能
    psi_t = psi_1d(ref_feat, Xt[:,0])                         # 模块04：监控漂移
    fire, why = trig.step(psi=psi_t, perf=perf, new_data_count=0)
    gate_result = '-'
    if fire:
        n_retrains += 1
        buf_X.append(Xt); buf_y.append(yt)
        Xall = np.vstack(buf_X[-3:]); yall = np.concatenate(buf_y[-3:])  # 重训+回放
        challenger = train_logistic(Xall, yall)
        # 模块03 门禁：challenger 在当前数据上是否显著优于 champion
        c_new = ((Xt @ challenger > 0).astype(float) == yt).astype(int)
        c_old = ((Xt @ champion   > 0).astype(float) == yt).astype(int)
        if bootstrap_better(c_new, c_old):
            champion = challenger                            # 晋升
            champion_hash = content_hash(np.round(champion,4).tolist())
            registry[champion_hash] = champion               # 模块02 版本
            n_promotions += 1; gate_result = 'PASS->晋升'
        else:
            gate_result = 'BLOCK->留守'
    tracking.append({'t':t,'perf':round(perf,3),'psi':round(psi_t,3),'champion':champion_hash})
    print(f'{t:<5}{perf:>8.3f}{psi_t:>8.3f}{("YES" if fire else "no"):>7}{gate_result:>10}{champion_hash:>11}')

print(f'\n12 周期内：重训 {n_retrains} 次，门禁放行晋升 {n_promotions} 次，版本库存 {len(registry)} 个模型')
assert n_retrains > 0, '漂移世界里应触发过重训'
assert n_promotions <= n_retrains, '晋升次数不超过重训次数(门禁拦掉了一些)'
assert len(tracking) == 12, '每周期都有追踪记录(模块01)'
assert len(registry) == n_promotions + 1, '版本库= 初始 + 每次晋升'
print('✅ 全生命周期闭环跑通：监控->触发->重训->门禁->版本/追踪，环转起来了')

**🧪 胶囊练习**：实现 `lifecycle_summary(tracking)`：从追踪记录里返回 `(平均性能, 最低性能, 用过的不同 champion 版本数)`。这就是一个生命周期的「健康报表」。

In [ ]:
def lifecycle_summary(tracking):
    # TODO: 返回 (mean perf, min perf, len(set(各周期 champion hash)))
    raise NotImplementedError

In [ ]:
# 自测
mean_p, min_p, n_versions = lifecycle_summary(tracking)
assert 0 <= min_p <= mean_p <= 1, '性能应在[0,1]且 min<=mean'
assert n_versions >= 1, '至少用过 1 个 champion 版本'
print(f'生命周期报表: 平均性能={mean_p:.3f}, 最低={min_p:.3f}, 用过 {n_versions} 个 champion 版本')
print('✅ 胶囊练习通过：生命周期健康报表')

In [ ]:
# 📖 胶囊参考答案
def lifecycle_summary(tracking):
    perfs = [r['perf'] for r in tracking]
    versions = set(r['champion'] for r in tracking)
    return (float(np.mean(perfs)), float(np.min(perfs)), len(versions))

---
## 🔧 旁注：这对应生产里的「持续训练(CT)」

你刚拼出的闭环，正是 Google MLOps 白皮书里 L1/L2 成熟度的核心——**CT（Continuous Training）**（伪代码，**本环境不跑**）：

```python
# 一条编排化的 CT pipeline（Airflow/Kubeflow 概念示意）
@pipeline
def continuous_training():
    if monitor.drift_or_perf_breach():        # 模块04 触发
        data = collect_and_label_feedback()    # 本模块：反馈+主动学习
        model = retrain(data, replay=old_data) # 本模块：重训+回放防遗忘
        if gate.passes(model, champion):       # 模块03 门禁
            registry.promote(model)            # 模块02 版本/晋升
        tracker.log(model)                     # 模块01 追踪
```

对应关系：整条 pipeline↔我们的闭环模拟、`retrain(replay=)`↔回放防遗忘、`gate.passes`↔bootstrap 门禁、`registry.promote`↔模块02 标签。这就是「训练完和上线后才是开始」的最终形态——一个能自我维持、随世界进化的系统。

### 小结 · 全课收官
- 反馈分**直接(高置信)/间接(有偏)**；间接信号别当硬标签(曝光偏差)；**主动学习**挑最不确定样本省标注。
- **重训触发**组合多条件 + 持续性 + 冷却防抖动；自动重训**必须**配自动门禁，否则等于自动事故。
- **灾难性遗忘**：增量学习会丢旧能力，用「旧测试集跌幅」量化，用 **replay** 抑制；拿不准就全量重训。
- **闭环 = 五模块协奏**：监控(04)触发→反馈+重训(05)产候选→门禁(03)把关→版本/追踪(01,02)留痕→上线→再监控。

🎓 **全课完结**。你从零造出了一套能跑的 ML 生命周期工具——这正是「训练完和上线后才是开始」的全部含义。回 [课程主页](../index.html) 复习，或翻 [术语词典](../glossary.md) / [参考清单](../references.md)。